# Programación declarativa @ GIA - URJC
## Curso 25-26
## Convocatoria extraordinaria
## Prueba 1

La duración de la prueba es de 1 hora y 20 minutos.

# Preámbulo

Correspondencia entre tipos algebraicos de datos y aritmética:

$$ 
\begin{array}{cc}
\mathrm{\bf Scala\ ADTs} & \mathrm{\bf Arithmetic} \\
\hline
\mathtt{Unit} & 1 \\
\mathtt{Nothing} & 0 \\
\mathtt{Either[X, Y]} & x + y \\
\mathtt{(X, Y)} & x * y \\
\mathtt{X => Y} & {y^x}  
\end{array}
$$

In [ ]:
object StandardTypes: 

    // Option[X] ~= Either[Unit, X]
    // |Option[X]| = 1 + |X|
    enum Option[+X]:
        case None extends Option[Nothing]
        case Some[X](x: X) extends Option[X]

Plantilla para demostrar isomorfismos:

In [ ]:
trait Isomorphic[A, B]:
    
    def from(a: A): B
    
    def to(b: B): A
    
    // equality 
    
    def equalA(a1: A, a2: A): Boolean = 
        a1 == a2
    
    def equalB(b1: B, b2: B): Boolean =
        b1 == b2
    
    // Bijection laws
    
    def law1(a: A): Boolean = 
        equalA(to(from(a)), a)
    
    def law2(b: B): Boolean = 
        equalB(from(to(b)), b)

Correspondencia de Curry-Howard entre tipos algebraicos de datos y lógica proposicional:

$$ 
\begin{array}{cc}
\mathrm{\bf Scala\ ADTs} & \mathrm{\bf Logic} \\
\hline
\mathtt{Unit} & \top \\
\mathtt{Nothing} & \bot \\
\mathtt{Either[P, Q]} & p \vee q \\
\mathtt{(P, Q)} & p \wedge q \\
\mathtt{P => Q} & p \rightarrow q \\
\mathtt{Not[P]} & \neg p \\
\mathtt{P <=> Q} & p \leftrightarrow q 
\end{array}
$$

In [ ]:
type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

In [ ]:
trait ExcludedMiddle: 
    def apply[P]: Either[P, Not[P]]

In [ ]:
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

# Ejercicio 1
__(2 puntos)__

Utiliza la correspondencia de Curry-Howard para demostrar las siguientes tautologías y argumentos de la lógica proposicional intuicionista: 

#### a) __(1 punto)__

##### $\vdash \neg\neg(p \vee \neg p)$

In [ ]:
def proof[P]: Not[Not[Either[P, Not[P]]]] =
    k => k(Right(p => k(Left(p))))

#### b) __(1 punto)__

##### $\{ p \rightarrow q,\ r \rightarrow s,\ \neg q \vee \neg s \} \vdash \neg p \vee \neg r$

In [ ]:
def proof[P, Q, R, S](f: P => Q, g: R => S, h: Either[Not[Q], Not[S]]): Either[Not[P], Not[R]] =
    h match
        case Left(nq)  => Left(p => nq(f(p)))
        case Right(ns) => Right(r => ns(g(r)))

# Ejercicio 2
__(2 puntos)__

Demuestra mediante la correspondencia de Curry-Howard los siguientes teoremas de la lógica clásica: 

#### a) __(1 punto)__

##### $\vdash ((p \rightarrow q) \rightarrow r) \rightarrow (r \rightarrow p) \rightarrow p$

(utilizando la ley de la doble negación para la proposición $p$)

In [ ]:
def proof[P, Q, R](DN: DoubleNegation): ((P => Q) => R) => (R => P) => P =
    h => g => DN[P](k => k(g(h(k))))

#### b) __(1 punto)__

##### $\vdash (\neg p \rightarrow q) \rightarrow (p \vee q)$

(utilizando la ley del tercio excluso para la proposición $p$)

In [ ]:
def proof[P, Q](EM: ExcludedMiddle): (Not[P] => Q) => Either[P, Q] =
    f => EM[P] match
        case Left(p) => Left(p)
        case Right(np) => Right(f(np))

# Ejercicio 3
__(1,5 puntos)__

#### __a) (0,5 puntos)__ 

¿Cuántos valores hay del siguiente tipo? 

`Option[Boolean] => Option[Boolean]`

Razona matemáticamente aplicando la correspondencia entre tipos algebraicos de datos y aritmética. 

$|Option[Boolean]| = 1 + |Boolean| = 1 + 2 = 3$, luego $|Option[Boolean] => Option[Boolean]| = |Option[Boolean]|^{|Option[Boolean]|} = 3^3 = 27$.

#### __b) (1 punto)__ 

Implementa todas las funciones del tipo `Option[Boolean] => Option[Boolean]` que sean **biyectivas** (es decir, isomorfismos de `Option[Boolean]` en sí mismo).

In [ ]:
// Hay 3! = 6 biyecciones (permutaciones de {None, Some(true), Some(false)})

val b1: Option[Boolean] => Option[Boolean] = 
    identity

val b2: Option[Boolean] => Option[Boolean] =
    case None    => None
    case Some(b) => Some(!b)

val b3: Option[Boolean] => Option[Boolean] =
    case None        => Some(true)
    case Some(false) => Some(false)
    case Some(true)  => None

val b4: Option[Boolean] => Option[Boolean] =
    case None        => Some(false)
    case Some(false) => None
    case Some(true)  => Some(true)

val b5: Option[Boolean] => Option[Boolean] =
    case None        => Some(true)
    case Some(false) => None
    case Some(true)  => Some(false)
    
val b6: Option[Boolean] => Option[Boolean] =
    case None        => Some(false)
    case Some(false) => Some(true)
    case Some(true)  => None

# Ejercicio 4
__(3 puntos)__

Se desea demostrar que se cumple el siguiente isomorfismo:

<p style="text-align: center;">$\mathtt{Option[Option[Option[Unit]]]} \cong \mathtt{(Boolean,\ Boolean)}$</p>

#### __a) (0,5 puntos)__ 

Realiza la demostración mostrando que los tipos involucrados en el isomorfismo tienen la misma cardinalidad (es decir, el mismo número de valores o instancias).

$|Option[Option[Option[Unit]]]| = 1 + (1 + (1 + 1)) = 4$ y $|(Boolean, Boolean)| = 2 \cdot 2 = 4$. Ambos tipos tienen 4 valores.

#### __b) (2 puntos)__ 

Realiza la demostración instanciando la plantilla `Isomorphic`, es decir, implementando las funciones `from` y `to` de dicha plantilla de tal forma que una sea la inversa de la otra.

In [ ]:
class Iso extends Isomorphic[Option[Option[Option[Unit]]], (Boolean, Boolean)]:

    def from(o: Option[Option[Option[Unit]]]): (Boolean, Boolean) = o match
        case None                 => (false, false)
        case Some(None)           => (false, true)
        case Some(Some(None))     => (true, false)
        case Some(Some(Some(()))) => (true, true)

    def to(b: (Boolean, Boolean)): Option[Option[Option[Unit]]] = b match
        case (false, false) => None
        case (false, true)  => Some(None)
        case (true, false)  => Some(Some(None))
        case (true, true)   => Some(Some(Some(())))

#### __c) (0,5 puntos)__ 

Demuestra que se cumplen las leyes del isomorfismo (`law1` y `law2`) para todos los valores de ambos tipos.

In [ ]:
val iso = Iso()
iso.law1(None)
iso.law1(Some(None))
iso.law1(Some(Some(None)))
iso.law1(Some(Some(Some(None))))
iso.law2((false, false))
iso.law2((false, true))
iso.law2((true, false))
iso.law2((true, true))

# Ejercicio 5 
__(1,5 puntos)__

Implementa la siguiente función mediante desarrollo dirigido por tipos, **mostrando cada paso en una celda independiente**: en cada celda escribe el estado de la implementación con los huecos (`??? : Tipo`) aún sin resolver, e indica mediante un comentario la constructora o destructora aplicada en ese paso. 

In [ ]:
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    ???

// INTRODUCE TU RESPUESTA MOSTRANDO CADA PASO EN UNA CELDA INDEPENDIENTE, INDICANDO EL TIPO DE CONSTRUCTORA O DESTRUCTORA APLICADA MEDIANTE UN COMENTARIO

In [ ]:
// Constructor (exponente)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        ??? : (A => Either[B, C] => C)

In [ ]:
// Constructor (exponente)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            ??? : (Either[B, C] => C)

In [ ]:
// Constructor (exponente)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                ??? : C

In [ ]:
// Destructor (suma)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => ??? : C
                    case Right(c) => ??? : C

In [ ]:
// Destructor (variable)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => ??? : C
                    case Right(c) => c

In [ ]:
// Destructor (función)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => g(??? : (A, B))
                    case Right(c) => c

In [ ]:
// Constructor (producto)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => g((??? : A, ??? : B))
                    case Right(c) => c

In [ ]:
// Destructor (variable)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => g((a : A, ??? : B))
                    case Right(c) => c

In [ ]:
// Destructor (variable)
def foo[A, B, C]: (((A, B)) => C) => A => Either[B, C] => C =
    (g: ((A, B)) => C) =>
        (a: A) =>
            (e: Either[B, C]) =>
                e match
                    case Left(b)  => g((a : A, b : B))
                    case Right(c) => c